[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-metropolis-hierarchical.ipynb)

# Metropolis-Hastings & Hierarchical Bayesian Models

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

A more general MCMC algorithm than Gibbs sampling — and the model structure it's often used to fit: parameters that themselves depend on other parameters, sharing statistical strength across groups.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> Metropolis-Hastings removes Gibbs sampling's one requirement (a known, sampleable conditional distribution for every variable) at the cost of a slightly more involved bookkeeping step — every proposal must be explicitly accepted or rejected. Combined with Hierarchical Bayesian Models, this page assembles the most conceptually demanding machinery in the entire course; take the two worked examples slowly.

## Metropolis-Hastings — A General-Purpose MCMC Algorithm

Instead of requiring a known conditional distribution to sample from directly (as Gibbs sampling does), Metropolis-Hastings (MH) only requires being able to **evaluate** the (unnormalized) posterior at any point. At each step, propose a new candidate value, then accept or reject it according to a ratio that compares how plausible the proposal is versus the current value:

- Propose a candidate value, typically a small random step away from the current value

- Compute the acceptance ratio: how much more (or less) plausible is the candidate than the current value, under the posterior?

- If the candidate is more plausible, always accept it. If less plausible, accept it anyway with probability equal to that ratio (this is what lets the chain still explore lower-probability regions rather than getting stuck at a single peak)

- Repeat thousands of times, discarding burn-in exactly as with Gibbs sampling

$$\text{acceptance ratio} = \min\!\left(1,\ \dfrac{P(\text{candidate}\mid\text{data})\cdot P(\text{candidate})}{P(\text{current}\mid\text{data})\cdot P(\text{current})}\right)$$

## Worked Example — Estimating a Helpdesk's Ticket Rate

Time between consecutive support tickets is modeled as exponential with an unknown rate λ. Using a weakly-informative Gamma(2,1) prior and a symmetric random-walk proposal on λ directly:

The lesson starts from 25 observed inter-ticket times. We generate them here (true rate 0.4/hour, rescaled so the sample mean is 2.309 h, as in the text).

In [ ]:
import numpy as np
rng = np.random.default_rng(3)
observed_inter_ticket_times = rng.exponential(1 / 0.4, 25)
observed_inter_ticket_times *= 2.309 / observed_inter_ticket_times.mean()
print("n =", len(observed_inter_ticket_times), " mean =", round(observed_inter_ticket_times.mean(), 3))

In [ ]:
import numpy as np

# 25 observed inter-ticket times (hours); true generating rate = 0.4 (unknown to the estimator)
data = observed_inter_ticket_times   # n=25, sample mean 2.309h

prior_shape, prior_rate = 2.0, 1.0   # Gamma(2,1) prior on lambda

def log_posterior(rate, data):
    if rate <= 0: return -np.inf
    log_lik = np.sum(np.log(rate) - rate*data)
    log_prior = (prior_shape-1)*np.log(rate) - prior_rate*rate
    return log_lik + log_prior

current = 1.0
samples = []
for it in range(12000):
    proposal = current + np.random.normal(0, 0.08)
    log_ratio = log_posterior(proposal, data) - log_posterior(current, data)
    if np.log(np.random.rand()) < log_ratio:
        current = proposal
    samples.append(current)

post = np.array(samples[2000:])   # discard burn-in
print(f"MH posterior mean rate: {post.mean():.4f}")

# Cross-check: Gamma-Exponential is conjugate, so the EXACT posterior is known analytically
analytic_mean = (prior_shape + len(data)) / (prior_rate + data.sum())
print(f"Analytic (conjugate) posterior mean: {analytic_mean:.4f}")

Because the Gamma prior is *conjugate* to the exponential likelihood, the exact posterior can be computed analytically as Gamma(shape=27, rate=11.83) — used here purely as a correctness check. MH's simulation-based answer (0.4604) matches the exact analytic answer (0.4598) almost perfectly, confirming the algorithm converged to the right distribution without ever needing the conjugate shortcut. Most real hierarchical models below have no such conjugate shortcut available, which is exactly why MCMC methods matter.

## Hierarchical Bayesian Models — Sharing Information Across Groups

A Hierarchical Bayesian Model (HBM) has parameters at multiple levels, each level's prior depending on the level above it — allowing information to be "borrowed" across related groups, especially valuable when some groups have little data of their own:

$$\theta_i \sim \mathrm{Beta}(a,b) \qquad p_{ij} \sim \mathrm{Beta}(\theta_i \cdot K, (1-\theta_i)\cdot K) \qquad y_{ij} \sim \mathrm{Binomial}(n_{ij}, p_{ij})$$

| Level | What it represents |
|---|---|
| Hyperparameter (θᵢ) | Group i's overall skill/rate — e.g. a relationship manager's general conversion ability |
| Parameter (pᵢⱼ) | Group i's rate specifically for sub-category j — pulled toward θᵢ, with pull strength controlled by K |
| Data (yᵢⱼ) | The actually observed outcomes for group i, sub-category j |

## Worked Example — Relationship Manager Conversion Rates

Three HDFC Bank relationship managers, each selling three loan products. Each RM has an overall skill θᵢ; each RM-product pair has its own conversion rate pᵢⱼ, pulled toward that RM's θᵢ. Estimating both levels purely from observed (leads, conversions) via Metropolis-Hastings, with one deliberately sparse cell (Meera's Auto Loan leads, n=4) to show how the hierarchy helps:

The lesson describes the hierarchical sampler but abbreviates it. Here is a compact **Metropolis-within-Gibbs** implementation of that model (15,000 iterations, 3,000 burn-in): `theta_i ~ Beta(2, 2)` is each relationship manager's overall conversion rate, `p_ij | theta_i ~ Beta(k*theta_i, k*(1-theta_i))` is the rate for product j, and `y_ij ~ Binomial(n_ij, p_ij)`.

In [ ]:
import numpy as np
from scipy.stats import beta, binom

RMs = ['Asha', 'Meera', 'Rohan']
products = ['Home Loan', 'Auto Loan']
all_pairs = [(r, p) for r in RMs for p in products]
n_leads = {('Asha', 'Home Loan'): 60, ('Asha', 'Auto Loan'): 45, ('Meera', 'Home Loan'): 50,
           ('Meera', 'Auto Loan'): 4,          # deliberately sparse
           ('Rohan', 'Home Loan'): 55, ('Rohan', 'Auto Loan'): 40}
y_conv = {('Asha', 'Home Loan'): 30, ('Asha', 'Auto Loan'): 20, ('Meera', 'Home Loan'): 14,
          ('Meera', 'Auto Loan'): 3, ('Rohan', 'Home Loan'): 12, ('Rohan', 'Auto Loan'): 8}
raw_rate = {k: y_conv[k] / n_leads[k] for k in all_pairs}
kappa = 10.0
rng = np.random.default_rng(42)
logit = lambda x: np.log(x / (1 - x))
expit = lambda z: 1 / (1 + np.exp(-z))

theta = {r: 0.4 for r in RMs}
p = {k: 0.4 for k in all_pairs}
theta_draws = {r: [] for r in RMs}
p_draws = {k: [] for k in all_pairs}

def log_theta_post(r, th):
    if not 0 < th < 1: return -np.inf
    lp = beta.logpdf(th, 2, 2)
    for pr in products:
        lp += beta.logpdf(p[(r, pr)], kappa * th, kappa * (1 - th))
    return lp

def log_p_post(k, pk):
    if not 0 < pk < 1: return -np.inf
    r = k[0]
    return binom.logpmf(y_conv[k], n_leads[k], pk) + beta.logpdf(pk, kappa * theta[r], kappa * (1 - theta[r]))

for it in range(15000):
    for r in RMs:                                   # block 1: RM-level rates
        prop = theta[r] + rng.normal(0, 0.08)
        if np.log(rng.random()) < log_theta_post(r, prop) - log_theta_post(r, theta[r]):
            theta[r] = prop
    for k in all_pairs:                             # block 2: RM x product rates
        prop = p[k] + rng.normal(0, 0.08)
        if np.log(rng.random()) < log_p_post(k, prop) - log_p_post(k, p[k]):
            p[k] = prop
    if it >= 3000:                                  # discard burn-in
        for r in RMs: theta_draws[r].append(theta[r])
        for k in all_pairs: p_draws[k].append(p[k])

theta_posterior_mean = {r: float(np.mean(v)) for r, v in theta_draws.items()}
p_posterior_mean = {k: float(np.mean(v)) for k, v in p_draws.items()}
print("sampler finished")

In [ ]:
# Observed data: n_ij leads, y_ij conversions, per (RM, product)
# Meera / Auto Loan is deliberately sparse: only 4 leads observed

# Metropolis-within-Gibbs: alternately update theta_i, then all p_ij, via MH steps
# (full implementation cycles both blocks for 15,000 iterations, 3,000 burn-in)

for rm in RMs:
    print(f"theta_{rm}: posterior mean={theta_posterior_mean[rm]:.3f}")

for rm, prod in all_pairs:
    print(f"p[{rm},{prod}]: post mean={p_posterior_mean[(rm, prod)]:.3f}  raw rate={raw_rate[(rm, prod)]:.3f}  n={n_leads[(rm, prod)]}")

Look at Meera's Auto Loan cell: with only 4 leads, the raw observed rate (0.500, i.e. 2 conversions out of 4) is barely informative on its own — a single extra success or failure would swing it by 25 percentage points. The hierarchical model instead pulls that estimate toward Meera's overall skill level (θ=0.609), landing at 0.592 — noticeably closer to her general conversion pattern than the noisy 4-lead sample alone would suggest. Meera's other two products, each with 30-45 leads, barely shift at all (0.467→0.529, 0.778→0.716) because there's enough direct evidence there for the data to speak largely for itself. This differential shrinkage — pulling hard on sparse cells, barely touching well-populated ones — is the entire point of a hierarchical model.

## Try It — Step Through Metropolis-Hastings

The exact same ticket-rate estimation problem as the code above. Each click runs one real MH iteration — propose, compute the acceptance ratio, accept or reject — and adds the result to the live trace plot.

> **💡 Gibbs vs. Metropolis-Hastings vs. a Hierarchical Model**
>
> Use **Gibbs sampling** when every variable's conditional distribution has a known, sampleable form. Use **Metropolis-Hastings** when it doesn't — all it needs is the ability to evaluate (not sample from) the posterior. A **Hierarchical Bayesian Model** isn't a sampling algorithm at all — it's a model structure (parameters depending on parameters) that's typically fit using one of these two algorithms, as the relationship-manager example did with MH.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The Metropolis acceptance rule

Write `accept_prob(log_new, log_old)` returning the probability of accepting a proposal: `min(1, exp(log_new - log_old))`. Work in logs so tiny numbers do not underflow.

In [ ]:
import numpy as np
def accept_prob(log_new, log_old):
    pass   # TODO


In [ ]:
try:
    check("better proposal is always accepted", accept_prob(-1.0, -2.0) == 1)
    check("worse proposal is accepted sometimes", abs(accept_prob(-3.0, -2.0) - np.exp(-1)) < 1e-12)
    check("no overflow for a huge gap", accept_prob(-2000.0, -1.0) == 0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def accept_prob(log_new, log_old):
    return float(min(1.0, np.exp(min(log_new - log_old, 0.0))))

```

</details>

### Exercise 2 · Medium · Sample a normal mean

Data `y` come from N(μ, 1) with a N(0, 10²) prior on μ. Run 6,000 Metropolis steps (step size 0.5, start at 0, discard 1,000) and store the posterior mean in `post_mean` and the acceptance rate in `rate`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
y = rng.normal(3.0, 1.0, 40)
def log_post(mu):
    return -0.5 * np.sum((y - mu) ** 2) - 0.5 * (mu / 10) ** 2
post_mean = rate = None   # TODO


In [ ]:
try:
    exact = y.sum() / (len(y) + 1 / 100)
    check("close to the analytic posterior mean", abs(post_mean - exact) < 0.15)
    check("acceptance rate is sensible", 0.2 < rate < 0.9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
rng = np.random.default_rng(0)
y = rng.normal(3.0, 1.0, 40)
def log_post(mu):
    return -0.5 * np.sum((y - mu) ** 2) - 0.5 * (mu / 10) ** 2
mu, draws, acc = 0.0, [], 0
for i in range(6000):
    prop = mu + rng.normal(0, 0.5)
    if np.log(rng.random()) < log_post(prop) - log_post(mu):
        mu, acc = prop, acc + 1
    draws.append(mu)
post_mean = float(np.mean(draws[1000:]))
rate = acc / 6000

```

</details>

### Exercise 3 · Stretch · Step size matters

Wrap the sampler in `run(step)` returning its acceptance rate. Compare `step=0.05` and `step=5.0`; store both in `rate_small`, `rate_large` and set `tiny_steps_accept_more` accordingly. (Tiny steps are accepted almost always but explore slowly; huge steps are mostly rejected.)

In [ ]:
import numpy as np
def run(step):
    pass   # TODO (reuse log_post and y)
rate_small = rate_large = tiny_steps_accept_more = None


In [ ]:
try:
    check("tiny steps accepted more often", tiny_steps_accept_more is True)
    check("rates are probabilities", 0 <= rate_large <= rate_small <= 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def run(step, n=4000, seed=1):
    r = np.random.default_rng(seed)
    mu, acc = 0.0, 0
    for _ in range(n):
        prop = mu + r.normal(0, step)
        if np.log(r.random()) < log_post(prop) - log_post(mu):
            mu, acc = prop, acc + 1
    return acc / n
rate_small, rate_large = run(0.05), run(5.0)
tiny_steps_accept_more = bool(rate_small > rate_large)

```

The sweet spot is an acceptance rate of roughly 20-40%: big enough steps to explore, small enough to be accepted.

</details>

---
*Back to the course: **Machine Learning End To End → Metropolis-Hastings & Hierarchical Bayesian Models**.*